# Random Forest / XGBoost Classifier

In [1]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import sys
sys.path.append('..')
from src.data_utils import stratified_split

/home/gabriel/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dados

In [2]:
df_principal = pd.read_csv('../data/clean_datasets/dataset_clean_final.csv')

train_df, val_df = stratified_split(df_principal, 'text', 'label', test_size=0.20, seed=2026)
test_df = pd.read_csv('../data/test_datasets/final_test_dataset.csv', sep=';')
test_df = test_df.rename(columns={'Label': 'label', 'Text': 'text'})

print(f"Tamanhos - Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")

Tamanhos - Treino: 240000 | Validação: 60000 | Teste: 315


## Encoding

In [3]:
print("A codificar as labels para números (para o XGBoost não se queixar)...")
le = LabelEncoder()
y_train = le.fit_transform(train_df['label'])
y_val = le.transform(val_df['label'])
y_test = le.transform(test_df['label'])

# Guardar o mapeamento
classes_nomes = le.classes_
print("Mapeamento das classes:", {i: label for i, label in enumerate(classes_nomes)})

A codificar as labels para números (para o XGBoost não se queixar)...
Mapeamento das classes: {0: 'Anthropic', 1: 'Google', 2: 'Human', 3: 'Meta', 4: 'OpenAI'}


## TF-IDF

In [4]:
MAX_FEATURES = 20000
tfidf = TfidfVectorizer(max_features=MAX_FEATURES)

X_train = tfidf.fit_transform(train_df['text'])
X_val = tfidf.transform(val_df['text'])
X_test = tfidf.transform(test_df['text'])

print("TF-IDF aplicado")

TF-IDF aplicado


## Treino

### Random Forest

In [5]:
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=2026, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# 4. Avaliar
preds = modelo_rf.predict(X_val)
acc = accuracy_score(y_val, preds)
print(f"Accuracy Real na Validação: {acc * 100:.2f}%")
print(classification_report(y_val, preds))

Accuracy Real na Validação: 82.05%
              precision    recall  f1-score   support

           0       0.84      0.91      0.88     12000
           1       0.83      0.78      0.81     12000
           2       0.80      0.85      0.83     12000
           3       0.78      0.81      0.79     12000
           4       0.85      0.75      0.79     12000

    accuracy                           0.82     60000
   macro avg       0.82      0.82      0.82     60000
weighted avg       0.82      0.82      0.82     60000



### XGBoost

In [6]:
search = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=2026, 
    n_jobs=-1,
    tree_method='hist'
)

# mlflow
with mlflow.start_run(run_name="XGBoost_Tuned"):

    search.fit(X_train, y_train)
    
        # Prever na Validação e no Teste
    preds_val = search.predict(X_val)
    preds_test = search.predict(X_test)
    
    acc_val = accuracy_score(y_val, preds_val)
    acc_test = accuracy_score(y_test, preds_test)
    
        # Registar tudo no MLflow automaticamente
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("max_features", MAX_FEATURES)
    mlflow.log_metric("val_acc", acc_val)
    mlflow.log_metric("test_acc", acc_test)
    
    print("\n" + "="*40)
    print(f"ACCURACY VALIDAÇÃO: {acc_val * 100:.2f}%")
    print(f"ACCURACY TESTE (FINAL): {acc_test * 100:.2f}%")
    print("="*40 + "\n")
    
    print("Classification Report do TESTE:")
    print(classification_report(y_test, preds_test, target_names=classes_nomes))
    
    # Guardar os modelos
    import joblib
    joblib.dump(search, '../modelos/xgboost_tuned.pkl')
    joblib.dump(tfidf, '../modelos/tfidf_xgb.pkl')
    joblib.dump(le, '../modelos/label_encoder_xgb.pkl')
    print("Modelos guardados na pasta 'modelos/'")

2026/04/02 00:58:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/02 00:58:17 INFO mlflow.store.db.utils: Updating database tables



ACCURACY VALIDAÇÃO: 79.00%
ACCURACY TESTE (FINAL): 35.24%

Classification Report do TESTE:
              precision    recall  f1-score   support

   Anthropic       0.25      0.68      0.37        53
      Google       0.39      0.22      0.28        51
       Human       0.81      0.36      0.50       116
        Meta       0.41      0.27      0.33        51
      OpenAI       0.14      0.18      0.16        44

    accuracy                           0.35       315
   macro avg       0.40      0.34      0.33       315
weighted avg       0.49      0.35      0.37       315

Modelos guardados na pasta 'modelos/'
